# AML_03_2 — Du score à l’action, puis du local à Kaggle

Conception pédagogique : Mathieu Weill.

Ce deuxième atelier se place **à côté d’AML_03_atelier.ipynb**. Il ne change pas ses cellules. Le PowerPoint « J3 — Suite accompagnée » suit les repères **B1 à B12** ci-dessous.

**QUID :** comment transformer des scores en actions justifiées, puis exécuter une expérience sur Kaggle et récupérer sa preuve ?

Les dix paiements de la première partie sont **fabriqués pour le cours** ; leurs chiffres ne sont pas des scores de la compétition. Les appels Kaggle sont désactivés par défaut et se déclenchent un par un avec le professeur. Aucun envoi de soumission n’est automatique.

## B1 — Dix paiements dont nous connaissons la réponse

Une ligne est un paiement. `fraude` est sa vraie classe dans cet exemple ; `score` est une sortie de modèle fictive. Un score élevé place le paiement plus haut dans le classement. Nous ne supposons pas ces scores calibrés.

In [ ]:
import sys, json, math, subprocess, shlex, hashlib
from pathlib import Path
import pandas as pd
import numpy as np
paiements=pd.DataFrame({'id':list('ABCDEFGHIJ'),
 'score':[.95,.80,.60,.45,.35,.20,.15,.10,.05,.02],
 'fraude':[1,0,1,1,0,0,1,0,0,0]})
display(paiements)
assert paiements.fraude.sum()==4 and len(paiements)==10
print('Quatre fraudes, six paiements normaux. Données pédagogiques fabriquées.')

## B2 — Une règle d’alerte explicite

Nous déclenchons une alerte si **score ≥ seuil**. Le choix de l’égalité fait partie du contrat. Au seuil 0,5, écrire d’abord les identifiants qui déclencheront une alerte.

In [ ]:
def bilan(seuil):
 alerte=paiements.score.ge(seuil)
 vrai=paiements.fraude.eq(1)
 tp=int((alerte & vrai).sum());fp=int((alerte & ~vrai).sum())
 fn=int((~alerte & vrai).sum());tn=int((~alerte & ~vrai).sum())
 return dict(seuil=seuil,alertes=int(alerte.sum()),TP=tp,FP=fp,FN=fn,TN=tn,
  rappel=tp/(tp+fn),precision=tp/(tp+fp) if tp+fp else float('nan'),
  accuracy=(tp+tn)/len(paiements))
print('Alertes à 0,5 :',paiements.loc[paiements.score.ge(.5),'id'].tolist())
display(pd.Series(bilan(.5)))
assert [bilan(.5)[k] for k in ['TP','FP','FN','TN']]==[2,1,2,5]

## B3 — Comparer trois seuils, sans réentraîner

Prévoir l’effet d’un seuil plus faible sur le nombre d’alertes, puis lire rappel et précision. Baisser le seuil augmente ou conserve les alertes et les vrais positifs sur ce tableau fixe ; la précision n’est pas forcément monotone.

In [ ]:
comparaison=pd.DataFrame([bilan(s) for s in [.5,.3,.1]])
display(comparaison.round(3))
assert comparaison.alertes.tolist()==[3,5,8]
assert comparaison.TP.tolist()==[2,3,4]
assert comparaison.FP.tolist()==[1,2,4]
assert comparaison.FN.tolist()==[2,1,0]

## B4 — Le classement et le seuil répondent à deux questions

L’AUC compare les scores des fraudes et des paiements normaux. Elle ne change pas ici lorsque nous déplaçons seulement le seuil d’alerte. Nous pouvons vérifier le calcul en comparant les 4 × 6 paires. Ce petit tableau ne mesure pas la performance Kaggle.

In [ ]:
from sklearn.metrics import roc_auc_score
positifs=paiements.loc[paiements.fraude.eq(1),'score']
negatifs=paiements.loc[paiements.fraude.eq(0),'score']
gagnes=sum(float(p>n)+.5*float(p==n) for p in positifs for n in negatifs)
auc=roc_auc_score(paiements.fraude,paiements.score)
print('Paires bien classées :',gagnes,'sur',len(positifs)*len(negatifs))
print('AUC :',round(auc,4))
assert gagnes==19 and abs(auc-19/24)<1e-12

## B5 — Ajouter un coût et une capacité

Hypothèses **fictives** : une fraude manquée coûte 100 unités ; une fausse alerte coûte 5 unités. Ce calcul suppose ces coûts identiques pour tous les paiements et n’évalue pas tout le service.

La capacité de trois vérifications change la décision : un seuil produisant huit alertes n’est pas immédiatement exploitable. Calculer aussi ce que repère le top 3.

In [ ]:
comparaison['cout_fictif']=100*comparaison.FN+5*comparaison.FP
display(comparaison[['seuil','alertes','TP','FP','FN','cout_fictif']])
assert comparaison.cout_fictif.tolist()==[205,110,20]
top3=paiements.nlargest(3,'score')
print('Capacité 3 :',top3.id.tolist(),'fraudes trouvées :',int(top3.fraude.sum()))
assert top3.fraude.sum()==2
print('Le coût le plus faible parmi ces essais n’annule pas la contrainte de capacité.')

## B6 — Identifier l’environnement et préparer la CLI Kaggle

Une API permet à un programme d’échanger avec un service. La CLI `kaggle` est le client officiel en ligne de commande. Elle ne transforme pas notre modèle en API de prédiction.

Avant de continuer, exécuter dans une cellule dédiée, au signal du professeur :

```python
%pip install "kaggle==2.2.5" "six>=1.17,<2"
```

Puis relancer B6. `six` est inclus explicitement : cette version du client l’importe. Utiliser Python 3.11 ou plus récent pour cette CLI. Si `%pip` n’est pas disponible, utiliser le terminal de l’environnement du notebook : `python -m pip install "kaggle==2.2.5" "six>=1.17,<2"`.

In [ ]:
import importlib.metadata as metadata
print('Python du notebook :',sys.executable)
print('Version Python :',sys.version.split()[0])
try:
 print('Kaggle installé :',metadata.version('kaggle'))
except metadata.PackageNotFoundError:
 print('CLI non installée dans ce noyau : suivre la consigne B6 au signal du professeur.')
# Laisser ce chemin par défaut après installation dans le noyau courant.
KAGGLE_PYTHON=sys.executable
# Si le professeur utilise une CLI isolée, il indique explicitement son Python.
ROOT=Path.cwd();RUN=ROOT/'runs_J3_2';RUN.mkdir(exist_ok=True)
print('Dossier de travail créé :',RUN.name)
print('Authentification, commande macOS/Linux :',shlex.join([KAGGLE_PYTHON,'-m','kaggle','auth','login']))
print('Authentification, commande PowerShell :',"& '"+KAGGLE_PYTHON.replace("'","''")+"' -m kaggle auth login")

## B7 — Authentification personnelle

Dans un terminal, copier la commande d’authentification adaptée à votre système, affichée par B6. Elle utilise exactement le Python du notebook. La forme générale est **`python -m kaggle auth login`**. Le client ouvre une autorisation dans le navigateur. Chacun vérifie son compte et accomplit lui-même cette étape. Ne pas projeter de jeton, de code d’autorisation ou de fichier d’identifiants.

Autre méthode si OAuth bloque : obtenir son propre jeton dans **https://www.kaggle.com/settings/api**, puis le saisir avec `getpass`, dans une cellule dédiée exécutée uniquement par son propriétaire :

```python
import getpass, os
os.environ['KAGGLE_API_TOKEN']=getpass.getpass('Jeton Kaggle personnel : ')
```

Ne jamais écrire le jeton en clair dans le notebook, les slides, une capture ou GitHub. Le jeton saisi dans le noyau est transmis aux sous-processus locaux par l’environnement ; il faudra le ressaisir après redémarrage. Les anciens fichiers `kaggle.json` existent encore mais ne sont pas indispensables à la méthode OAuth.

**Aucune cellule ci-dessous n’affiche les identifiants.** Ne pas exécuter une commande qui imprime toute la configuration ou toutes les variables d’environnement.

In [ ]:
def kaggle_cmd(*args,execute=False):
 commande=[KAGGLE_PYTHON,'-m','kaggle',*args]
 # Les arguments proposés ne contiennent aucun secret.
 print('Commande :',shlex.join(commande))
 if not execute:
  print('Non exécutée. Mettre execute=True uniquement au signal du professeur.')
  return None
 return subprocess.run(commande,check=True)
kaggle_cmd('--version')
kaggle_cmd('kernels','list','--mine')

## B8 — Vérifier l’accès à la compétition avant de télécharger

Compétition de travail : **https://www.kaggle.com/competitions/playground-series-s6e11**.

Ouvrir sa page et lire les règles. Si une adhésion/acceptation est requise, chacun la réalise lui-même dans le navigateur ; la CLI ne remplace pas cet accord. Un refus d’accès peut aussi venir du compte, de l’état de la compétition ou d’un service indisponible.

Changer uniquement `execute=False` en `execute=True`, une commande à la fois, lorsque le professeur le demande.

In [ ]:
COMPETITION='playground-series-s6e11'
kaggle_cmd('competitions','files',COMPETITION,execute=False)

## B9 — Télécharger d’abord un petit fichier

Commencer par `sample_submission.csv`, après avoir confirmé son nom dans la liste. Cela teste le trajet sans recharger tout le train. La commande suivante écrit dans un dossier neuf de cet atelier et n’utilise pas `--force`.

Les options de la CLI installée se vérifient avec `python -m kaggle competitions download --help`. Les ZIP officiels déjà reçus restent utilisables si le téléchargement échoue ; cela ne compte pas comme un accès API réussi.

In [ ]:
DATA=RUN/'donnees';DATA.mkdir(exist_ok=True)
kaggle_cmd('competitions','download',COMPETITION,'-f','sample_submission.csv',
 '-p',str(DATA),execute=False)
fichiers=sorted(p.name for p in DATA.iterdir())
print('Fichiers réellement présents :',fichiers)
# Pas d’affirmation de réussite tant que la commande n’a pas été exécutée et contrôlée.
import zipfile
candidats=list(DATA.glob('sample_submission.csv'))
if candidats:
 fichier=candidats[0];sample=pd.read_csv(fichier)
else:
 fichier=next(iter(DATA.glob('*.zip')),None)
 if fichier:
  with zipfile.ZipFile(fichier) as archive:
   noms=[x for x in archive.namelist() if Path(x).name=='sample_submission.csv']
   assert len(noms)==1,'Archive sans fichier attendu unique.'
   with archive.open(noms[0]) as flux:sample=pd.read_csv(flux)
if fichier:
 assert list(sample.columns)==['id','is_fraud']
 print('Colonnes vérifiées :',sample.columns.tolist(),'lignes :',len(sample))
 print('SHA-256 du fichier reçu :',hashlib.sha256(fichier.read_bytes()).hexdigest())
else:
 print('Le contrôle de contenu attend un fichier réellement téléchargé.')

## B10 — Préparer une petite sonde privée sur Kaggle

Avant d’envoyer le vrai atelier ML, vérifier la boucle avec un notebook très court : calculer 2 + 2 et écrire `rapport.json`. Aucun GPU, aucune source de données, aucun appel Internet dans ce notebook distant. Il vérifie le trajet du code et des fichiers, pas la qualité d’un modèle.

Remplacer `votre-pseudo` par son identifiant Kaggle public, pas par un jeton ni nécessairement son nom d’affichage. Le slug utilisé est propre à cet atelier. Vérifier qu’il ne désigne pas un notebook important existant : `push` peut créer **ou mettre à jour** la référence indiquée.

In [ ]:
PSEUDO='votre-pseudo'
SLUG='mllab-j3-2-sonde'
REF=f'{PSEUDO}/{SLUG}'
PROBE=RUN/'sonde';PROBE.mkdir(exist_ok=True)
probe_code="""import json,platform
from pathlib import Path
report={'calcul':2+2,'python':platform.python_version(),'preuve':'sonde pédagogique'}
assert report['calcul']==4
Path('rapport.json').write_text(json.dumps(report,indent=2))
print('rapport.json créé ; calcul vérifié :',report['calcul'])
"""
probe={'cells':[{'cell_type':'code','id':'sonde','metadata':{},'source':[probe_code],
 'outputs':[],'execution_count':None}],
 'metadata':{'kernelspec':{'name':'python3','display_name':'Python 3','language':'python'}},
 'nbformat':4,'nbformat_minor':5}
(PROBE/'sonde.ipynb').write_text(json.dumps(probe,indent=2))
meta={'id':REF,'title':'Mllab J3 2 Sonde','code_file':'sonde.ipynb',
 'language':'python','kernel_type':'notebook','is_private':True,
 'enable_gpu':False,'enable_internet':False,'dataset_sources':[],
 'competition_sources':[],'kernel_sources':[]}
(PROBE/'kernel-metadata.json').write_text(json.dumps(meta,indent=2))
assert meta['is_private'] and not meta['enable_gpu']
print('Préparé localement :',REF)
print('Deux fichiers : sonde.ipynb et kernel-metadata.json. Aucun envoi effectué.')

## B11 — Envoyer, suivre, récupérer : trois actions distinctes

Le professeur fait vérifier les métadonnées et la référence avant d’autoriser chaque étape. **`kernels push` envoie le code et tente de l’exécuter** ; ce n’est pas une soumission au classement de la compétition. Une nouvelle exécution peut consommer les ressources du compte, même en CPU.

1. Changer son pseudo dans B10 et relancer B10.
2. Mettre `LANCER_SONDE=True` ci-dessous pour un envoi volontaire.
3. Consulter l’état : ne pas envoyer une nouvelle version à chaque fois que le calcul attend.
4. Récupérer les sorties après une réussite. Pour une comparaison reproductible, noter et utiliser la version exacte du notebook distant.

In [ ]:
LANCER_SONDE=False
if LANCER_SONDE:
 assert PSEUDO!='votre-pseudo','Renseigner son propre pseudo et relancer B10.'
 assert REF==f'{PSEUDO}/{SLUG}'
 kaggle_cmd('kernels','push','-p',str(PROBE),execute=True)
else:
 kaggle_cmd('kernels','push','-p',str(PROBE),execute=False)
# Exécuter séparément au signal, puis attendre un état terminé avant le téléchargement.
kaggle_cmd('kernels','status',REF,execute=False)
SORTIES=RUN/'retour_kaggle';SORTIES.mkdir(exist_ok=True)
kaggle_cmd('kernels','output',REF,'-p',str(SORTIES),execute=False)
report_path=SORTIES/'rapport.json'
if report_path.exists():
 rapport=json.loads(report_path.read_text());assert rapport['calcul']==4
 print('Sortie réellement récupérée et vérifiée :',rapport)
else:
 print('Aucun rapport distant récupéré pour le moment.')

## B12 — Passer de la sonde à la vraie expérience ML

La sonde ne mesure aucun score ML. Après cette boucle réussie, reprendre **`templates/KAGGLE_AML_03.ipynb`** du kit J3/J4 : la variante complète embarque le laboratoire et adapte les chemins. Elle possède deux cellules d’amorçage de plus que le notebook local.

Ajouter les données officielles autorisées sur Kaggle. Pour automatiser le transfert, créer un dossier distinct, y placer cette variante et ses propres métadonnées avec `competition_sources: ["playground-series-s6e11"]`. Garder l’accès privé, CPU au départ et les réglages du protocole local. Ne pas transférer un notebook corrigé ni des identifiants.

Comparer empreinte des identifiants, folds, limite de données, métrique et paramètres avant de comparer les scores. Les versions logicielles et certains algorithmes peuvent modifier les résultats. Un écart n’est pas une preuve de meilleur matériel.

Il est aussi possible d’importer le notebook dans l’interface Kaggle et de lancer Save Version. La voie graphique reste une solution si la CLI est indisponible ; noter alors que la boucle API n’a pas été complétée.

In [ ]:
journal={'question':'Vérifier la boucle local → Kaggle → local',
 'preuve_locale':{'probe_preparee':(PROBE/'sonde.ipynb').exists(),
                  'rapport_distant_present':report_path.exists()},
 'reference_kaggle':REF,'score_public':'non mesuré par cet atelier',
 'prochaine_etape':'Exécuter la variante ML avec le même protocole, si la sonde réussit.'}
(RUN/'journal.json').write_text(json.dumps(journal,ensure_ascii=False,indent=2))
print('Journal local écrit. Distinguer préparation, envoi, exécution et récupération.')

## Exercice individuel — votre décision et votre preuve

1. Avec une capacité de trois vérifications, pourquoi le seuil qui coûte le moins dans notre exemple peut-il être inexploitable ?
2. Quelle différence entre `kernels push`, `kernels output` et une soumission au classement ?
3. Un fichier local `rapport.json` suffit-il à prouver une exécution sur Kaggle ? Quelle provenance conserver ?
4. Quelle condition garder fixe pour attribuer un écart de score à un changement de modèle ?

Écrire une justification avant la correction collective. Les réponses réservées ne figurent pas dans cette version.

## Références

- https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md
- https://github.com/Kaggle/kaggle-cli/blob/main/docs/competitions.md
- https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md
- https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels_metadata.md
- https://scikit-learn.org/stable/modules/classification_threshold.html